# Phase 1 SFT on Kaggle — `post-training-lab`

Full SFT of Qwen2.5-0.5B-Instruct on a 5k slice of UltraChat-200k with QLoRA,
then re-runs the same eval harness with the trained adapter so `results/metrics.json`
ends up with a second row directly comparable to the Phase 0 baseline.

Flow: smoke (50 steps, no push) → full run (~80 steps, pushes adapter to HF Hub)
→ post-SFT eval (appends `sft_v1` row to `metrics.json`).

**Before you start**
1. Right sidebar → **Settings**:
   - **Accelerator** → `GPU T4 x2` (free, generous) or `GPU P100`.
   - **Internet** → `On`. _Required for the Hub push, dataset download, model download._
   - **Persistence** → `Variables and Files`.
2. Top bar → **Add-ons → Secrets**, add:
   - `HF_TOKEN` — must have **write scope** (not just read) so the trained adapter can be pushed. Mint at `huggingface.co/settings/tokens`.
   - `WANDB_API_KEY` — optional, only if you want training logs in W&B.
3. Edit the clone cell's `GITHUB_USER` and (if different) the `output.hub_repo` in `configs/sft_qwen05b.yaml`.
4. **Use `Save Version → Save & Run All` for the full run** — interactive kernels die after ~30 min idle. Full SFT + eval is ~3 hours wall-clock on T4; detached mode (9h cap) is the right tool.

Reference: `PROJECT.md` §6 Phase 1 (deliverable + success criterion), §4.4 (HF Hub naming), §5.4 (smoke-test discipline).

## 1. GPU sanity check

Fail fast if the accelerator isn't selected — SFT on CPU is hours-per-step.

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
GITHUB_USER = "agaonker"
REPO        = "post-training-lab"
BRANCH      = "main"
REPO_PATH   = f"/kaggle/working/{REPO}"

import os, shutil
os.chdir("/kaggle/working")
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone --depth 1 --branch {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git {REPO_PATH}
os.chdir(REPO_PATH)
!git log -1 --oneline

## 3. Point HF cache at the working dir

Same in-session/cross-session story as the baseline notebook — `/kaggle/working` persists across cell re-runs; **Save Version → Save & Run All** snapshots it as a re-attachable input dataset.

In [ ]:
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

## 4. Install the project

Pulls trl, peft, transformers, datasets, accelerate, bitsandbytes (CUDA-only,
so the Mac-side install path is the one in pyproject's platform marker — Kaggle
is Linux so we get bnb here). The `[eval]` extra brings in `lm-eval` for the
post-SFT evaluation step.

In [ ]:
!pip install -q -e .[eval]

## 5. Inject secrets — HF token MUST have write scope

Reads from Kaggle's Secrets panel (Add-ons → Secrets). The Hub push at the end
of `make sft` will fail with a 401 if the token is read-only — mint a fresh
fine-grained token with **write** access to your namespace before running this.

In [ ]:
from kaggle_secrets import UserSecretsClient
us = UserSecretsClient()

for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = us.get_secret(key)
        print(f"{key:14s} loaded")
    except Exception:
        print(f"{key:14s} not set")

# huggingface_hub picks up HF_TOKEN automatically. The training script's
# trainer.push_to_hub() call uses it for both repo creation and the upload.
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"

## 6. Patch dtype to fp16 on T4 / P100

Same logic as the baseline notebook: T4 (Turing) and P100 (Pascal) emulate
bf16 in software at ~half speed. Free Kaggle gives one of these by default,
so this almost always fires. Skip on V100 / A100 / H100 / L4.

In [ ]:
import subprocess, re, pathlib
gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()
print("GPU:", gpu_name)

if "T4" in gpu_name or "P100" in gpu_name:
    p = pathlib.Path("configs/base.yaml")
    p.write_text(re.sub(r"dtype:\s*bfloat16", "dtype: float16", p.read_text()))
    print("Patched configs/base.yaml -> dtype: float16 (native fp16 beats emulated bf16).")
    # bnb 4-bit compute dtype must match for consistency.
    p = pathlib.Path("configs/base.yaml")
    p.write_text(
        re.sub(r"bnb_4bit_compute_dtype:\s*bfloat16", "bnb_4bit_compute_dtype: float16", p.read_text())
    )
    print("Patched bnb_4bit_compute_dtype -> float16 for consistency.")
else:
    print("Native bf16 GPU — leaving configs untouched.")

## 7. SFT smoke (50 steps, no Hub push)

Catches wiring failures before paying full wall-clock. PROJECT.md §5.4: *"Never
launch [a long] run on code you haven't run 50 steps of on free tier."* On a
T4 with QLoRA, 50 steps is ~5 minutes. If this fails — dataset format error,
OOM, peft target_modules mismatch — fix here, not in the full run.

In [ ]:
!make sft-smoke

## 8. Full SFT run + push to HF Hub

5000 UltraChat samples × 1 epoch × effective batch 16 ≈ ~80 steps. T4
wall-clock with QLoRA is roughly 10–20 minutes. On success, the adapter is
pushed to `<output.hub_repo>` from `configs/sft_qwen05b.yaml` (default:
`agaonker/atlas-sft-qwen05b-v1`).

The summary JSON at the end is the same shape as `run_eval`'s — capture it
if the cell output gets truncated by a disconnect (Phase 0's lesson).

In [ ]:
!make sft

## 9. Post-SFT eval — append the Phase 1 row to `metrics.json`

Re-runs the same eval harness used for Phase 0, but with `--adapter` pointing
at the adapter we just pushed. Same tasks, same metric extraction, same JSON
schema — apples-to-apples by construction (the whole reason `atlas.eval.harness`
exists as a single entrypoint).

`--name sft_v1 --method sft` distinguishes the row from the `base` row. The
harness now writes incrementally to `results/metrics.sft_v1.partial.json` and
promotes to `metrics.json` only on success, so a mid-run disconnect won't
lose completed tasks (the Phase 0 lesson, now baked in).

Edit `HUB_ADAPTER` below if you customized `output.hub_repo` in the YAML.

In [ ]:
HUB_ADAPTER = "agaonker/atlas-sft-qwen05b-v1"

!python -m atlas.eval.harness \
    --config configs/baseline.yaml \
    --name sft_v1 --method sft \
    --adapter {HUB_ADAPTER}

## 10. Inspect the updated `metrics.json`

Two rows should now be present: `base` (Phase 0) and `sft_v1` (Phase 1). The
Phase 1 success criterion per PROJECT.md §6 is *"SFT model beats base on
IFEval by a clear margin"* — concretely, with the Phase 0 numbers in hand:
**IFEval prompt-strict > 0.25 and inst-strict > 0.40**.

In [ ]:
import json, pathlib
doc = json.loads(pathlib.Path("results/metrics.json").read_text())
for run in doc["runs"]:
    print(f"\n--- {run['name']} ({run['method']}) ---")
    for k, v in run["metrics"].items():
        if "sample_len" not in k:
            print(f"  {k:50s} {v:.4f}")

## 11. Get `metrics.json` back to your laptop

- **Quick**: right sidebar → **Output** tab → `working/post-training-lab/results/metrics.json` → ⋯ → Download. Save to your local `results/metrics.json` (overwrites the Phase-0-only file with the new two-row version) and commit.
- **Snapshot (recommended)**: **File → Save Version → Save & Run All**. The committed version archives `/kaggle/working/` — re-attachable, downloadable via `kaggle kernels output`, and persistent across sessions.